In [ ]:
import os
import sys 
from pyprojroot import here
sys.path.insert(0, str(here()))
from os.path import exists

import geopandas as gpd
from laos_gggi.data_functions import load_emdat_data, load_shapefile, load_rivers_data
from laos_gggi.data_functions.disaster_point_data import (load_disaster_point_data, 
                    load_synthetic_non_disaster_points, load_grid_point_data, load_non_disaster_grid)

from laos_gggi.plotting import configure_plot_style
from laos_gggi.statistics import get_distance_to_rivers, prediction_to_gpd_df , set_plotting_data, add_data, add_country_effect
from laos_gggi.data_functions.combine_data import load_all_data
from pymc.model.transform.optimization import freeze_dims_and_data

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import arviz as az
import scipy
import nutpie
import pathlib

import pymc as pm
import pytensor.tensor as pt
from laos_gggi.sample import sample_or_load


from laos_gggi.transformers import Standardize
from laos_gggi.data_functions.world_bank_data_loader import load_wb_data

configure_plot_style()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from laos_gggi.data_functions import load_ocean_heat_data
from laos_gggi.const_vars import OCEAN_HEAT_URL

In [ ]:
df_ocean = pd.read_csv(OCEAN_HEAT_URL, header=0, names=["Date", "Temp"])
df_ocean.Date = pd.to_datetime(df_ocean.Date, format="%Y-%m")
df_ocean.set_index("Date", inplace=True)
df_ocean = df_ocean.resample("YE").mean()
df_ocean.reset_index(inplace=True)
df_ocean["Date"] = df_ocean["Date"] - pd.offsets.YearBegin()
df_ocean.set_index("Date", inplace=True)

# Load and prepare data

In [ ]:
world = load_shapefile('world')
# rivers = load_rivers_data()
laos = world.query('ISO_A3 == "LAO"')

# Select SEA shape
region_list = [
    'LAO',  # Laos
    'VNM',  # Vietnam
    'KHM',  # Cambodia
    'MMR',  # Myanmar
    'THA'   # Thailand
]


# Define maps
country_maps = {}

for country in region_list:
    country_maps[country] = world.query('ISO_A3 == @country')


region_map = world.query('ISO_A3 in @region_list')
laos_map = world.query('ISO_A3 == "LAO"')

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
region_point_grid = load_grid_point_data(region='custom', grid_size=400, 
                            force_reload = True, 
                            iso_list=region_list,
                           file_reg_name="asean_list_a" ).rename(columns = {"lon": "long"})

region_point_grid_training = load_grid_point_data(region='custom', grid_size=200, 
                            force_reload = True, 
                            iso_list=region_list,
                           file_reg_name="asean_list_a" ).rename(columns = {"lon": "long"})

country_point_grids = {}

laos_point_grid = load_grid_point_data(region='laos',
                                       grid_size=400, 
                                       force_reload = False,).rename(columns = {"lon": "long"})

In [ ]:
all_data = load_all_data()
panel_data  = all_data["df_panel"][['population_density', 'gdp_per_cap', 'Population', 'precip', "real_gdp"]]
co2 = all_data["df_time_series"]["co2"]

In [ ]:
# Load disasters and non-disasters
disasters = load_disaster_point_data()
# not_disasters = load_synthetic_non_disaster_points(by='country', multiplier=3)

not_disasters = load_synthetic_non_disaster_points(countries= region_list, 
                                                   list_name= 'asean_a',
                                                   by='country',
                                                   multiplier=15, 
                                                   force_generate = True)
    

# Merge data frames
merged_df = pd.concat([not_disasters.assign(is_disaster = 0), 
                       disasters.reset_index().assign(is_disaster=15)], 
                      ignore_index= True)

# Adjust date format
merged_df["Start_Year"] = pd.to_datetime(merged_df["Start_Year"])

In [ ]:
disasters.query('ISO in @region_list').shape

In [ ]:
not_disasters.shape

In [ ]:
from statsmodels.tsa.seasonal import STL

precipitation = all_data["gpcc"]
precip_deviation = precipitation.groupby('ISO').transform(lambda x: x - x.iloc[:30].mean()).rename(columns={'precip':'precip_deviation'})

df_clim = all_data["df_time_series"][["co2", "Temp", "precip"]].iloc[1:-1].dropna(subset=['Temp'])
trend =  STL(pd.DataFrame(df_clim["Temp"].dropna()), period=3).fit().trend
dev_from_trend_ocean_temp = (df_clim['Temp'] - trend).to_frame(name='dev_ocean_temp')

In [ ]:
from functools import reduce
df = reduce(lambda l, r: pd.merge(l, r, left_on=['ISO', 'Start_Year'], right_index=True, how='left'), [merged_df, panel_data, precip_deviation])
df = reduce(lambda l, r: pd.merge(l, r , left_on=['Start_Year'], right_index=True, how='left'), [df, co2, dev_from_trend_ocean_temp])

In [ ]:
#Creating log variables
log_list = ["distance_to_river", "distance_to_coastline", "Total_Affected", "Total_Damage_Adjusted", "population_density",
            "gdp_per_cap"]
for y in log_list:
    df[f"log_{y}"] = np.log(df[y])



In [ ]:
#Delimiting data set
columns_to_use = ['ISO', 'Start_Year', "is_disaster", 'distance_to_river', 'distance_to_coastline',
               "Population", "co2", "precip_deviation", "dev_ocean_temp", "lat", "long" , "geometry", "real_gdp"]

features = ['log_distance_to_river', 'log_distance_to_coastline',
             "Population", "co2", "precip_deviation", "dev_ocean_temp", 'log_population_density',
             'log_gdp_per_cap', ]

model_df = df[ list( set( columns_to_use).union(set(features)) )]


In [ ]:
# Define list of features standardized
features_stand = []
for feature in features:
    features_stand.append(feature + "__standardized" )

time_varying_features = ['Population','co2','precip_deviation','dev_ocean_temp','log_population_density','log_gdp_per_cap',]
time_varying_features_stand = []

for feature in time_varying_features:
    time_varying_features_stand.append(feature + "__standardized")

### Create region and country data sets

In [ ]:
#Create the geodata set for sea disasters
region_disasters = model_df.query('ISO in @region_list & is_disaster == 1')

region_disasters_geo = gpd.GeoDataFrame(
                region_disasters,
    geometry=gpd.points_from_xy(region_disasters["long"], region_disasters["lat"]), crs="EPSG:4326"
            )

# Create the geodata set for countries disasters
country_disasters = {}
country_disasters_geo = {}

for country in region_list:
    country_disasters[country] = model_df.query('ISO == @country & is_disaster == 1')
    country_disasters_geo[country] = gpd.GeoDataFrame(
                    country_disasters[country],
        geometry=gpd.points_from_xy(country_disasters[country]["long"], country_disasters[country]["lat"]), crs="EPSG:4326"
                )

In [ ]:
# Define dfs
region_df = model_df.query('ISO in @region_list')

#Tranform dfs to geopandas df
region_df = gpd.GeoDataFrame(region_df,  geometry=gpd.points_from_xy(region_df["long"],
                                region_df["lat"]),crs="EPSG:4326")

country_df = {}
for country in region_list:
    country_df[country] = model_df.query('ISO == @country')
    country_df[country] = gpd.GeoDataFrame(country_df[country],  geometry=gpd.points_from_xy(country_df[country]["long"],
                                country_df[country]["lat"]),crs="EPSG:4326")


In [ ]:
# Merge geospatial data with time varying data

region_2020_data =  region_df.query('Start_Year == "2020-01-01"').iloc[[0]][time_varying_features + ["ISO", "real_gdp"]]
region_2016_data =  region_df.query('Start_Year == "2016-01-01"').iloc[[0]][time_varying_features + ["ISO", "real_gdp"]]


# We create the merged region_point_grid_extended for predictions
country_2020_data = {}
country_2020_data_df = {}
for country in region_list:
    country_2020_data[country] = region_df.query('Start_Year == "2020-01-01" & ISO == @country').iloc[[0]][time_varying_features + ["ISO", "real_gdp"]]
    country_2020_data_df[country] = pd.DataFrame()
    country_2020_data_df[country] = pd.concat([country_2020_data_df[country], country_2020_data[country]])

country_2016_data = {}
country_2016_data_df = pd.DataFrame()

country_2016_data = {}
country_2016_data_df = {}
for country in region_list:
    country_2016_data[country] = region_df.query('Start_Year == "2016-01-01" & ISO == @country').iloc[[0]][time_varying_features + ["ISO", "real_gdp"]]
    country_2016_data_df[country] = pd.DataFrame()
    country_2016_data_df[country] = pd.concat([country_2016_data_df[country], country_2016_data[country]])

In [ ]:
# Merge points with world ISO
region_point_grid_extended = {}
region_point_grid_extended_df = gpd.sjoin(region_point_grid, region_map, how="left", )
region_point_grid_extended_df = region_point_grid_extended_df.rename(columns = {"lon": "long"})


region_point_grid_extended["2020"] = pd.merge(region_point_grid_extended_df, region_2020_data, how = "left", left_on= "ISO_A3", right_on="ISO" ) 
region_point_grid_extended["2016"] = pd.merge(region_point_grid_extended_df, region_2016_data, how = "left", left_on= "ISO_A3", right_on="ISO" ) 

In [ ]:
# country_point_grid = {}

# for country in region_list:
#     country_point_grid[country]['ISO'] = 

# # Creating the laos_point_grid_extended
# laos_point_grid["ISO"] = "LAO"
# laos_point_grid_extended = {}
# laos_point_grid_extended_df = gpd.sjoin(laos_point_grid, laos_map, how="left", )
# laos_point_grid_extended_df = laos_point_grid_extended_df.rename(columns = {"lon": "long"})
# laos_point_grid_extended["2020"] = pd.merge(laos_point_grid_extended_df, country_2020_data_df, how = "left", left_on= "ISO", right_on="ISO" ) 
# laos_point_grid_extended["2015"] = pd.merge(laos_point_grid_extended_df, country_2015_data_df, how = "left", left_on= "ISO", right_on="ISO" ) 


In [ ]:
# drop NaNs

region_df = region_df.dropna()

In [ ]:
# Standardize SEA data
transformer_stand_ =  Standardize().fit(region_df)
region_df_stand = transformer_stand_.transform(region_df)
# lao_df_stand = transformer_stand_.transform(lao_df)

# other dfs
region_point_grid_extended["2020"] = transformer_stand_.transform(region_point_grid_extended["2020"]) 
region_point_grid_extended["2016"] = transformer_stand_.transform(region_point_grid_extended["2016"]) 

# laos_point_grid_extended["2020"] = transformer_stand_.transform(laos_point_grid_extended["2020"]) 
# laos_point_grid_extended["2016"] = transformer_stand_.transform(laos_point_grid_extended["2016"])

In [ ]:
# define terms to square and dfs
df_list =  [region_point_grid_extended["2020"], region_point_grid_extended["2016"], 
            # laos_point_grid_extended["2020"],
            # laos_point_grid_extended["2016"],
            region_df_stand, 
            # lao_df_stand
           ]

terms_to_square = ["log_gdp_per_cap__standardized", "log_population_density__standardized", 
                  # "log_distance_to_river__standardized", "log_distance_to_coastline__standardized"
                  ]

terms_to_square_squared = [x + "__squared" for x in terms_to_square]

# Add the sqaured versions of terms
for df_ in df_list:
    for x, y in zip(terms_to_square, terms_to_square_squared):
        df_[y] = df_[x] ** 2


# Adjust features_stand and  time_varying_features_stand
features_stand =  features_stand 
time_varying_features_stand = time_varying_features_stand 
# + terms_to_square_squared

In [ ]:
# grid cols
grid_cols = ['ISO_A3', 'long', 'lat', 'log_distance_to_river','log_distance_to_coastline', 
 'geometry', ]

In [ ]:
# Save files
region_point_grid_extended["2020"].to_csv(here("data/asean_region_point_grid.csv"))
# laos_point_grid_extended["2020"].to_csv(here("data/laos_point_grid.csv"))
region_df.to_csv(here("data/asean.csv"))
# lao_df.to_csv(here("data/lao.csv"))
region_df_stand.to_csv(here("data/asean_region_df_stand.csv"))
# lao_df_stand.to_csv(here("data/lao_df_stand.csv"))  

### ASEAN maps

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=144)
world.query('ISO_A3 in @region_list ').plot(facecolor='tab:blue', alpha=0.25, ax=ax)
region_point_grid_extended_df.query('distance_to_river > 1000').plot('is_island', ax=ax , markersize=0.1)
ax.set_xticks([])
ax.set_yticks([])

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from adjustText import adjust_text

# fig, ax = plt.subplots(figsize=(14, 9), dpi=144)

# # Plot the map and disaster locations
# world.query('ISO_A3 == "LAO"').plot(facecolor='tab:blue', alpha=0.25, ax=ax)
# world.query('ISO_A3 == "LAO"').plot(facecolor='none', edgecolor='k', lw=0.25, ax=ax)
# country_disasters[count_geo.query('is_disaster == 1').plot('is_disaster', ax=ax)

# texts = []
# for idx, row in country_disasters[count_geo.query('is_disaster == 1').iterrows():
#     # Compute an angle for rotation (in radians)
#     angle = np.random.uniform(0, 2 * np.pi)  # Random angle between 0 and 360 degrees

#     # Define the label distance from the point
#     r = 0.05  # Adjust this to control the distance from the point

#     # Convert polar coordinates to Cartesian (offset x, y)
#     dx = r * np.cos(angle)
#     dy = r * np.sin(angle)

#     texts.append(ax.annotate(row['Start_Year'].year, 
#                              xy=(row.geometry.x, row.geometry.y),  # Point location
#                              xytext=(row.geometry.x + dx, row.geometry.y + dy),  # Rotated position
#                              textcoords="data",
#                              fontsize=6, color='black',
#                              ha='center', va='center'))  # Keep text aligned

# # Adjust text positions slightly to avoid overlapping
# adjust_text(texts, ax=ax, only_move={'text': 'xy'}, expand_text=(1.1, 1.1));
# ax.set_xticks([])
# ax.set_yticks([])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9), dpi=144)
world.query('ISO_A3 in @region_list ').plot(facecolor='tab:blue', alpha=0.25, ax=ax)
world.query('ISO_A3 in @region_list').plot(facecolor='none', edgecolor='k', lw=0.25, ax=ax)
# rivers.plot(edgecolor='dodgerblue', lw=0.5, ax=ax)
# region_point_grid.plot( markersize=0.1, 
#                                       ax=ax, )

ax.axis('off')
# plt.title("Synthetic vs. real data")
plt.show()

# Model on the SEA data set: HSGP component

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9), dpi=144)
world.query('ISO_A3 in @region_list ').plot(facecolor='tab:blue', alpha=0.25, ax=ax)
world.query('ISO_A3 in @region_list').plot(facecolor='none', edgecolor='k', lw=0.25, ax=ax)
# rivers.plot(edgecolor='dodgerblue', lw=0.5, ax=ax)

region_df.plot('is_disaster', 
                                      markersize=1, 
                                      ax=ax, 
                                      legend=True, 
                                      # categorical=True,
                                      cmap='viridis')
ax.axis('off')
plt.title("Synthetic vs. real data")
plt.show()

In [ ]:
#Define cooords
is_disaster_idx , is_disaster = pd.factorize(region_df["is_disaster"])
ISO_idx, ISO = pd.factorize(region_df["ISO"]) 
obs_idx = region_df.index
gp_features = ["lat", "long"]

#Creating idx
xr_idx = xr.Coordinates.from_pandas_multiindex(region_df.set_index(['ISO', 'Start_Year']).index, 'obs_idx')

#Set coords
coords_sea = {"is_disaster" : is_disaster,
        "obs_idx": obs_idx,
        "ISO": ISO,
        "feature": features_stand,
        "gp_feature":gp_features }

# Model on the SEA data set: Full model

In [ ]:
region_df_stand.shape

In [ ]:
with pm.Model(coords=coords_sea) as model_sea_full:
    #Declare data
    X, Y= add_data(features= features_stand ,  target = "is_disaster", df =  region_df_stand, )
    ISO_idx_pt = pm.Data("ISO_idx_pt", ISO_idx, dims= ["obs_idx"] )
    
    # # #Country effect
    country_effect = pm.Normal("country_effect", mu = 0, sigma =1, dims = ["ISO"])

    # #Betas
    beta_sigma = [0.1] * 8
    beta = pm.Normal("beta", mu = 0, sigma = beta_sigma, dims = ["feature"])

    # HSGP process
    X_gp = pm.Data("X_gp", region_df[["lat", "long"]])

    # Prior on the HSGP
    eta = pm.Exponential("eta", scale=2)
    ell_params = pm.find_constrained_prior(
        pm.Lognormal, lower=0.5, upper=10.0, mass=0.95, init_guess={"mu": 1.0, "sigma": 1.0}
    )
    ell = pm.Lognormal("ell", **ell_params, dims=["gp_feature"])
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=2, ls=ell)

    m0, m1, c = 35, 35, 1.5
    gp = pm.gp.HSGP(m=[m0, m1], c=c, cov_func=cov_func)

    phi, sqrt_psd = gp.prior_linearized(X=X_gp)

    basis_coeffs = pm.Normal("basis_coeffs", size=gp.n_basis_vectors)
    HSGP_component = pm.Deterministic("HSGP_component", phi @ (basis_coeffs * sqrt_psd),dims= ["obs_idx"])


    #Model mu
    mu = pm.Deterministic("mu", 
                          country_effect[ISO_idx_pt] + 
                          X@beta +
                          HSGP_component , dims= ["obs_idx"] )
    

    # Now define the observed variable separately
    y_hat = pm.Bernoulli('y_hat', logit_p=mu, observed=Y, dims=['obs_idx'])

#Set the prior predictions
with freeze_dims_and_data(model_sea_full):
   prior_idata = pm.sample_prior_predictive( compile_kwargs = {"mode":"JAX"})

prior_idata.prior_predictive["y_hat"] = prior_idata.prior_predictive["y_hat"].astype(int)

az.plot_ppc(prior_idata, group = "prior", observed = prior_idata.observed_data  );

In [ ]:
if exists("asean_0.idata"):
    asean_idata = az.from_netcdf("asean_0.idata")

else:
    compiled_model = nutpie.compile_pymc_model(freeze_dims_and_data(model_sea_full), backend="jax", gradient_backend='jax')
    asean_idata = nutpie.sample(compiled_model, chains = 6, draws = 400, target_accept = 0.9)
    # Save the idata
    az.to_netcdf(data=asean_idata, filename=pathlib.Path("asean_0.idata"))


### SEA predictions

In [ ]:
az.summary(asean_idata, var_names=["beta", "eta", "eta_log__", "country_effect"] )

In [ ]:
# full model SEA predictions

# Rebuild ISO_idx_sea
ISO_to_idx = {name: idx for idx, name in enumerate(ISO)}
ISO_idx_sea=  region_point_grid_extended['2020'].ISO.map(ISO_to_idx.get)

asean_idata_plot = {}

for year in ["2020",]:
    with model_sea_full.copy() as temp_model:
        #Declare data
        pm.set_data({"X_gp":region_point_grid_extended[year][["lat", "long"]],
                     "Y": np.full(region_point_grid_extended[year].shape[0], 0 ),
                     "X": region_point_grid_extended[year][features_stand],
                     "ISO_idx_pt": ISO_idx_sea
     
                },
            coords= {"obs_idx": region_point_grid_extended[year].index.values } 
           )
    
        y_hat_invlogit = pm.Deterministic('y_hat_invlogit', pm.math.invlogit(temp_model["y_hat"] ))

        HSGP_component_invlogit = pm.Deterministic('HSGP_component_invlogit', pm.math.invlogit(temp_model["HSGP_component"] ))
    
    
    
    with freeze_dims_and_data(temp_model):
        asean_idata_plot[year] = pm.sample_posterior_predictive(asean_idata, var_names=["y_hat_invlogit","y_hat", "HSGP_component",
                                                                                                          "HSGP_component_invlogit",], 
                                                             )


In [ ]:
# Create the geopandas version of the predictions
model_sea_full_predictions_geo = {}

for year in ["2020",]:
    model_sea_full_predictions_geo[year] = prediction_to_gpd_df(prediction_idata = asean_idata_plot[year] , 
                         variables = ["y_hat_invlogit", "y_hat" ,"HSGP_component","HSGP_component_invlogit"  ] , 
                         points = region_point_grid_extended['2020'] )

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 4), dpi= 144 )

sns.histplot( data = model_sea_full_predictions_geo["2020"]["y_hat"]["y_hat"], bins = 30
             , ax = ax,  )
ax.set_xlabel("Value", fontsize=14)  # X-axis label
ax.set_ylabel("Frequency", fontsize=14)  # Y-axis label
ax.tick_params(axis='both', which='major', labelsize=12)  # Tick labels
# fig.savefig(here("notebooks/final_paper/figures/event_full_neigh_hist.png"));

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(figsize=(10, 6), dpi= 144 )
model_sea_full_predictions_geo["2020"]["y_hat"].plot("y_hat",legend=True, ax=ax,markersize =1.5, vmax = 0.05 )
region_disasters_geo.plot(ax=ax, alpha = 0.3, c = "r", markersize =0.7, )
# plt.title("Event probability for Laos neighbors", );


ax.set_xticks([])
ax.set_yticks([])
# fig.savefig(here("notebooks/final_paper/figures/event_full_neigh_pred.png"));

### Laos predictions

In [ ]:
# Exclude point outside Lao

country_disasters[count_geo= country_disasters[count_geo.query('log_distance_to_coastline > 4')

In [ ]:
# full model SEA predictions

# Rebuild ISO_idx_sea
ISO_to_idx = {name: idx for idx, name in enumerate(ISO)}
ISO_idx_laos =  laos_point_grid_extended['2020'].ISO.map(ISO_to_idx.get)
asean_idata_plot_laos = {}

for year in ["2020", "2015"]:
    with model_sea_full.copy() as temp_model:
        #Declare data
        pm.set_data({"X_gp": laos_point_grid[["lat", "long"]],
                     "Y": np.full(laos_point_grid_extended['2020'].shape[0], 0 ),
                     "X": laos_point_grid_extended[year][features_stand],
                     "ISO_idx_pt": ISO_idx_laos
     
                },
            coords= {"obs_idx": laos_point_grid_extended['2020'].index.values } 
           )
    
        y_hat_invlogit = pm.Deterministic('y_hat_invlogit', pm.math.invlogit(temp_model["y_hat"] ))

        # River effect
        river_effect = pm.Deterministic('river_effect', temp_model["beta"][0] * laos_point_grid_extended[year]['log_distance_to_river__standardized'])
        # Coast effect
        coast_effect = pm.Deterministic('coast_effect',  temp_model["beta"][1] * laos_point_grid_extended[year]['log_distance_to_coastline__standardized'])
                                                
    
    with freeze_dims_and_data(temp_model):
        asean_idata_plot_laos[year] = pm.sample_posterior_predictive(
            asean_idata, var_names=["y_hat_invlogit","HSGP_component", "y_hat", "river_effect", "coast_effect"], 
                                                         )


In [ ]:
# Create the geopandas version of the predictions
model_sea_full_predictions_geo_laos = {}

for year in ["2020", "2015"]:
    model_sea_full_predictions_geo_laos[year] = prediction_to_gpd_df(prediction_idata = asean_idata_plot_laos[year] , 
                         variables = ["y_hat_invlogit","HSGP_component", "y_hat", "river_effect",'coast_effect'] , 
                         points = laos_point_grid_extended['2020'] )

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(10, 4), dpi= 144 )

var_list = ["HSGP_component", "river_effect",'coast_effect']
legen_list = [False, False, True]

for var, n in zip(var_list, [0,1,2] ):
    model_sea_full_predictions_geo_laos["2020"][var].plot(var,legend= legen_list[n], ax = axes[n] 
                                                     ,markersize =1.5, vmax = 0.6 )

    country_disasters[count_geo.plot(ax= axes[n], alpha = 0.3, c = "r", markersize =2, )
    axes[n].set_xticks([])
    axes[n].set_yticks([])
    axes[n].set_title(var)

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(figsize=(6, 3), dpi= 144 )
model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=True, ax=ax,markersize =1, vmax = 0.07,)
country_disasters[count_geo.plot(ax=ax, alpha = 0.3, c = "r", markersize =2, )
# plt.title("Event probability for Laos", )

ax.set_xticks([])
ax.set_yticks([])
# fig.savefig(here("notebooks/final_paper/figures/event_full_laos_pred.png"));

In [ ]:
# Plot for 2015


#Plot the predictions
fig, ax = plt.subplots(figsize=(6, 3), dpi= 144 )
model_sea_full_predictions_geo_laos["2015"]["y_hat"].plot("y_hat",legend=True, ax=ax,markersize =1, vmax = 0.07,)
country_disasters[count_geo.plot(ax=ax, alpha = 0.3, c = "r", markersize =2, )
# plt.title("Event probability for Laos", )

ax.set_xticks([])
ax.set_yticks([])
# fig.savefig(here("notebooks/final_paper/figures/event_full_laos_pred.png"));

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(figsize=(6, 3), dpi= 144 )
model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=False, ax=ax,markersize =1, vmax = 0.05,)
laos_point_grid.loc[[4135]].plot(ax=ax, c = "r", markersize =12, marker="*" )
plt.title("Event probability for Laos", );

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(1,2,figsize=(10, 6), dpi= 144 )
model_sea_full_predictions_geo["2020"]["y_hat"].plot("y_hat",legend=False, ax=ax[0] ,markersize =1.5, vmax = 0.1 )
region_disasters_geo.plot(ax=ax[0] , alpha = 0.3, c = "r", markersize =0.7, )
# plt.title("Event probability for Laos neighbors", );
# ax[0].set_title( "Vietnam, Laos, Thailand and Cambodia", size = 16)

model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=True, ax=ax[1],markersize =1.5, vmax = 0.1,)
country_disasters[count_geo.plot(ax=ax[1], alpha = 0.3, c = "r", markersize =2, )

# ax[1].set_title( "Laos", size = 16)

for x in [0,1]:
    ax[x] .set_xticks([])
    ax[x] .set_yticks([])

# fig.savefig(here("notebooks/final_paper/figures/event_combined .png"));

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(1,2,figsize=(10, 6), dpi= 144 )
model_sea_full_predictions_geo["2020"]["y_hat"].plot("y_hat",legend=False, ax=ax[0] ,markersize =1.5, vmax = 0.05 )
region_disasters_geo.plot(ax=ax[0] , alpha = 0.3, c = "r", markersize =0.7, )
# plt.title("Event probability for Laos neighbors", );
ax[0].set_title( "Vietnam, Laos, Thailand and Cambodia", size = 16)

model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=True, ax=ax[1],markersize =1.5, vmax = 0.05,)
country_disasters[count_geo.plot(ax=ax[1], alpha = 0.3, c = "r", markersize =2, )

ax[1].set_title( "Laos", size = 16)

for x in [0,1]:
    ax[x] .set_xticks([])
    ax[x] .set_yticks([])

# fig.savefig(here("notebooks/final_paper/figures/event_full_neigh_pred.png"));

### 2025 Lao predictions

We now import the file with 2025 predictions

In [ ]:
predictions_geo = {}
for x in [2025, 2035]:
    predictions_geo[x] = pd.read_csv(here(f'data/prediction_files/geo_pred_{x}'))
    predictions_geo[x] = gpd.GeoDataFrame(predictions_geo[x],  geometry=gpd.points_from_xy(predictions_geo[x]["long"],
                                predictions_geo[x]["lat"]),crs="EPSG:4326")


In [ ]:
# 2020 results
fig, ax = plt.subplots(1,1,figsize=(10, 6), dpi= 144 )
model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=True, ax=ax,markersize =1.5, vmax = 0.1,)

model_sea_full_predictions_geo_laos["2020"]["y_hat"].sort_values(by = 'y_hat').tail(10).plot(ax=ax, alpha = 0.3, c = "r", markersize =5, )
# country_disasters[count_geo.plot(ax=ax[1], alpha = 0.3, c = "r", markersize =2, )

ax.set_title( "Laos", size = 16)

ax .set_xticks([])
ax .set_yticks([])

# fig.savefig(here("notebooks/final_paper/figures/event_full_neigh_pred.png"));

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(1,1,figsize=(10, 6), dpi= 144 )
predictions_geo[2025].plot("y_hat",legend=True, ax=ax,markersize =1.5, vmax = 0.1,)

country_disasters[count_geo.plot(ax=ax, alpha = 0.3, c = "r", markersize =2, )

# ax.set_title( "Laos", size = 16)

ax .set_xticks([])
ax .set_yticks([]);
# fig.savefig(here("notebooks/final_paper/data/figures/lao_pred_2025.png"));

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(1,1,figsize=(10, 6), dpi= 144 )
predictions_geo[2035].plot("y_hat",legend=True, ax=ax,markersize =1.5, vmax = 0.1,)

country_disasters[count_geo.plot(ax=ax, alpha = 0.3, c = "r", markersize =2, )

ax.set_title( "Laos", size = 16)

ax .set_xticks([])
ax .set_yticks([]);


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(10, 6), dpi= 144 )

var_list = ["HSGP_component", "river_effect",'coast_effect']
legen_list = [False, False, False]

for var, n in zip(var_list, [0,1,2] ):
    predictions_geo[2025].plot(var,legend= legen_list[n], ax = axes[n] 
                                                     ,markersize =1.5, vmax = 0.05 )

    country_disasters[count_geo.plot(ax= axes[n], alpha = 0.3, c = "r", markersize =2, )
    axes[n].set_xticks([])
    axes[n].set_yticks([])
    axes[n].set_title(var)

# fig.savefig(here("notebooks/final_paper/data/figures/lao_pred_decomp_2025.png"));

In [ ]:
# time change
fig, axes = plt.subplots(1,3,figsize=(10, 6), dpi= 144 )


model_sea_full_predictions_geo_laos["2015"]["y_hat"].plot("y_hat",legend=False, ax=axes[0] ,markersize =1.5, vmax = 0.05,)

country_disasters[count_geo.plot(ax=axes[0], alpha = 0.3, c = "r", markersize =2, )


legend_list = [False, False, True]
for year, n in zip([2025, 2035], [1,2]):
    predictions_geo[year].plot("y_hat",legend=legend_list[n], ax=axes[n],markersize =1.5, vmax = 0.1,)
    country_disasters[count_geo.plot(ax=axes[n], alpha = 0.3, c = "r", markersize =2, )

# Set ticks and titles for each subplot
for year, n in zip([2015, 2025, 2035], [0, 1, 2]):
    axes[n].set_xticks([])
    axes[n].set_yticks([])
    axes[n].set_title(str(year))

# fig.savefig(here("notebooks/final_paper/data/figures/laos_pred_time_change.png"));

### Cities maps

In [ ]:
cities_loc = pd.read_csv(here('data/cites_loc.csv'), index_col=0 )

cities_loc = gpd.GeoDataFrame(cities_loc,  geometry=gpd.points_from_xy(cities_loc["long"],
                                cities_loc["lat"]),crs="EPSG:4326")

In [ ]:
#Plot the predictions
fig, ax = plt.subplots(1,1,figsize=(14, 8), dpi= 144 )
model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=True, ax=ax,markersize =1.5, vmax = 0.05,)

cities_loc.plot( c="r", marker='*', markersize=25, linestyle='-', ax=ax)
# country_disasters[count_geo.plot(ax=ax[1], alpha = 0.3, c = "r", markersize =2, )

ax.set_title( "Laos", size = 16)

ax .set_xticks([])
ax .set_yticks([]);

# fig.savefig(here("notebooks/final_paper/figures/event_full_neigh_pred.png"));

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

# Plot the predictions
fig, ax = plt.subplots(1, 1, figsize=(14, 6), dpi=144)

# Example plot (replace with your actual plotting code)
predictions_geo[2025].plot("y_hat",legend=False, ax=ax,markersize =1.5, vmax = 0.1,)

# Plot the city locations
cities_loc.plot(ax=ax, kind='scatter', x='long', y='lat', c='r', marker='*', s=250, edgecolor='k')

# Add city names as labels with white border
for idx, row in cities_loc.iterrows():
    ax.annotate(
        row.name,  # City name (assuming the index is the city name)
        (row["long"], row["lat"]),  # Coordinates for annotation
        textcoords="offset points",
        xytext=(5, 5),  # Offset for better visibility
        ha='left',
        fontsize=10,
        color='black',
        fontweight="bold",
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")]
    )

# Remove ticks
ax.set_xticks([])
ax.set_yticks([]);
ax.set_xlabel('')
ax.set_ylabel('');

# fig.savefig(here("notebooks/final_paper/data/figures/city_loc.png"));

In [ ]:
# Plot the predictions
fig, ax = plt.subplots(1, 2, figsize=(14, 6), dpi=144)

model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat",legend=False, ax= ax[0],markersize =1.5, vmax = 0.05,)

cities_loc.plot( c="r", marker='*', markersize=25, linestyle='-', ax= ax[0])



model_sea_full_predictions_geo_laos["2020"]["y_hat"].plot("y_hat", legend=True, ax=ax[1], markersize=1.5, vmax=0.05)

# Plot the city locations
cities_loc.plot(c="r", marker='*', markersize=25, linestyle='-', ax=ax[1])

# Add city names as labels
for idx, row in cities_loc.iterrows():
    ax[1].annotate(
        row.name,  
        (row["long"], row["lat"]),  # Coordinates for annotation
        textcoords="offset points",
        xytext=(5, 5),  
        ha='left',
        fontsize=10,
        color='black',
        fontweight="bold",
        path_effects=[path_effects.withStroke(linewidth=2, foreground="white")]
    )
for x in [0,1]:
    # ax[x].set_title("Laos", size=16)
    ax[x].set_xticks([])
    ax[x].set_yticks([])
